In [1]:
import os
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv


load_dotenv()
openai_client = OpenAI()

In [2]:
INSTRUCTIONS_PROMPT = """
You are an expert NLP data annotator.
Generate a batch of {batch_size} highly diverse and realistic sentences containing mentions of mountains, 
peaks, hills, or mountain ranges. 

The dataset style should match real-world natural text, including:
1. Short personal updates (e.g., "Just reached the summit of Mt. Rainier!").
2. Standard geographic facts / encyclopedic sentences (e.g., "The Andes Mountains extend across South America.").
3. Outdoor recreation and hiking blog snippets (e.g., "When hiking through the Carpathian Mountains, make sure 
to bring extra water.").

For each sentence, you must provide:
1. The raw sentence `text`.
2. A list of character `spans` marking the EXACT start and end index (0-indexed, start-inclusive, end-exclusive) of 
each mountain name entity within that exact `text`.
3. The raw `entity_names` corresponding to those spans.

Make sure to include some sentences with multiple mountain names and others with single names. 
Ensure absolute accuracy of the spans. Double-check your character counting. For example, in "I climbed Mt Fuji.", 
"Mt Fuji" starts at index 10 and ends at 17, so the span is [10, 17]. Ensure no off-by-one errors.
"""

In [3]:
from pydantic import BaseModel, Field
from typing import List, Tuple

class MountainRecord(BaseModel):
    text: str = Field(
        ..., 
        description="The complete raw sentence containing one or more mountain names."
    )
    
    markers: List[List[int]] = Field(
        ..., 
        description="List of character index pairs [start_char_idx, end_char_idx] pointing exactly to the mountain names."
    )
    
    entity_names: List[str] = Field(
        ..., 
        description="The exact text string(s) of the mountain names extracted by the spans (e.g., ['Mount Rainier'])."
    )


class MountainBatch(BaseModel):
    records: List[MountainRecord] = Field(
        ..., 
        description="A collection of annotated mountain sentences."
    )

In [5]:
def llm_api_call(prompt:str, model:str = "gpt-5.4-mini"):
    response = openai_client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful synthetic data generation assistant."},
            {"role": "user", "content": prompt}
        ],
        response_format=MountainBatch
    )
    return response.choices[0].message.parsed, response.usage

In [6]:
def correct_markers(text: str, entity_names: List[str], markers: List[List[int]]) -> List[List[int]]:
    """
    Programmatically self-corrects marker indices by anchoring around the LLM's 
    suggested spans, safeguarding against overlapping spans and substring mismatches.
    """
    corrected_markers = []
    for i, name in enumerate(entity_names):
        llm_suggested_start = markers[i][0] if i < len(markers) else 0
        
        # Locate all occurrences of this name in the text
        occurrences = []
        start = 0
        while True:
            idx = text.find(name, start)
            if idx == -1:
                break
            occurrences.append(idx)
            start = idx + 1
            
        if occurrences:
            # Anchor to the occurrence closest to the LLM's recommended index
            best_start = min(occurrences, key=lambda x: abs(x - llm_suggested_start))
            # Append a list instead of a tuple
            corrected_markers.append([best_start, best_start + len(name)])
        else:
            # Fallback to the original span if name is completely missing
            if i < len(markers):
                corrected_markers.append(markers[i])
                
    return corrected_markers

In [7]:
def generate_synthetic_mountains(total_records=1000, batch_size=50):
    all_records = list()
    all_usages = list()
    num_batches = total_records // batch_size
    
    print(f"Starting generation of {total_records} records in {num_batches} batches...")
    
    for batch_idx in tqdm(range(num_batches)):
        try:
            
            instructions_prompt = INSTRUCTIONS_PROMPT.format(batch_size=batch_size)
            batch_data, usage = llm_api_call(prompt=instructions_prompt)
            all_usages.append(usage)
            
            # Apply marker correction
            for record in batch_data.records:
                markers = correct_markers(record.text, record.entity_names, record.markers)
                
                all_records.append({
                    "text": record.text,
                    "marker": markers
                })
                
            # Respect rate limits between batches
            time.sleep(1.0)
            
        except Exception as e:
            print(f"\nError generating batch {batch_idx + 1}: {e}")
            continue
            
    df_synthetic = pd.DataFrame(all_records)
    return df_synthetic, all_usages

In [8]:
df_synthetic, usages = generate_synthetic_mountains()

os.makedirs("../data", exist_ok=True)
df_synthetic.to_csv("data/synthetic_dataset.csv", index=False)
print(f"\nSuccessfully generated and saved {len(df_synthetic)} valid records!")

Starting generation of 1000 records in 20 batches...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [04:04<00:00, 12.24s/it]


Successfully generated and saved 1007 valid records!


AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'

In [12]:
def calc_usage_cost(usage):
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    input_cost = (usage.prompt_tokens / 1_000_000) * input_price_per_million
    output_cost = (usage.completion_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


def calc_total_costs(usages):
    total_cost = 0.0

    for usage in usages:
        cost = calc_usage_cost(usage)
        total_cost = total_cost + cost["total_cost"]

    return total_cost

In [13]:
total_costs = calc_total_costs(usages)
print(f"\nTotal costs: {total_costs:.2f} $$$")

print("\nExample data:")
print(df_synthetic.head(3))


Total costs: 0.15 $$$

Example data:
                                                text      marker
0            Just reached the summit of Mt. Rainier!  [[27, 38]]
1   The Andes Mountains extend across South America.   [[4, 19]]
2  When hiking through the Carpathian Mountains, ...  [[24, 44]]
